# FLUX.1 Colab Image Asset Generation Pipeline

此 Notebook 專為在 **Google Colab (A100 / L4 / T4)** 上執行 **FLUX.1 [schnell / dev]** 高畫質視覺資產生成而設計。
可獨立產出高品質角色、場景概念圖，亦可直接作為後續 LTX-2.5 影片生成的第一影格（首格圖）視覺資產。

## 1. 檢查 GPU 硬體與儲存空間

In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), "錯誤：未偵測到 GPU，請至「執行階段 > 變更執行階段類型」選擇 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
free_disk_gb = round(shutil.disk_usage('/content').free / 2**30, 1)

print(f"[GPU] {gpu_name} (VRAM: {vram_gb} GiB)")
print(f"[CUDA] {torch.version.cuda}")
print(f"[可用磁碟空間] {free_disk_gb} GiB")

## 2. 安裝推論依賴環境

In [ ]:
!pip install -q --upgrade pip
!pip install -q "diffusers>=0.30.0" "transformers>=4.44.0" accelerate sentencepiece torchvision huggingface_hub protobuf
print("FLUX 依賴安裝完成。")

## 3. Hugging Face 認證與快取

In [ ]:
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face 授權完成。")

cache_dir = "/content/cache_flux"
use_drive_cache = os.environ.get("USE_DRIVE_CACHE", "1") == "1"
if use_drive_cache:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_path = Path('/content/drive/MyDrive/models/FLUX.1')
        drive_path.mkdir(parents=True, exist_ok=True)
        cache_dir = str(drive_path)
        print(f"啟用 Google Drive 快取: {cache_dir}")
    except Exception as e:
        print(f"略過 Drive 掛載: {e}")
        os.makedirs(cache_dir, exist_ok=True)
else:
    os.makedirs(cache_dir, exist_ok=True)

## 4. 讀取任務參數與提示詞

In [ ]:
prompt_file = os.environ.get("FLUX_PROMPT_FILE", "/content/prompt.txt")
output_path = os.environ.get("FLUX_OUTPUT_PATH", "/content/output.png")
width = int(os.environ.get("FLUX_WIDTH", "1280"))
height = int(os.environ.get("FLUX_HEIGHT", "720"))
steps = int(os.environ.get("FLUX_STEPS", "4")) # schnell 預設 4 步
seed = int(os.environ.get("FLUX_SEED", "42"))
model_name = os.environ.get("FLUX_MODEL", "black-forest-labs/FLUX.1-schnell")

if os.path.exists(prompt_file):
    with open(prompt_file, "r", encoding="utf-8") as f:
        prompt = f.read().strip()
else:
    prompt = os.environ.get("FLUX_PROMPT", "Cinematic concept art of a futuristic astronaut cat inside a spaceship cockpit, high detail, 8k")

print("=== 生圖任務配置 ===")
print(f"Prompt: {prompt}")
print(f"解析度: {width}x{height}")
print(f"推論步數: {steps}")
print(f"隨機種子: {seed}")
print(f"模型: {model_name}")
print(f"輸出路徑: {output_path}")

## 5. 載入 FLUX 模型並生成視覺資產

In [ ]:
import torch
from diffusers import FluxPipeline

print(f"正在載入 {model_name} (請稍候)...\n")
pipe = FluxPipeline.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    cache_dir=cache_dir
)
pipe.enable_sequential_cpu_offload()

generator = torch.Generator(device="cuda").manual_seed(seed)

print("開始生圖推論...")
image = pipe(
    prompt=prompt,
    width=width,
    height=height,
    num_inference_steps=steps,
    guidance_scale=0.0 if "schnell" in model_name else 3.5,
    max_sequence_length=256 if "schnell" in model_name else 512,
    generator=generator
).images[0]

Path(output_path).parent.mkdir(parents=True, exist_ok=True)
image.save(output_path)
print(f"\n視覺資產生成完畢！已儲存至: {output_path}")

## 6. 驗證產出成品

In [ ]:
assert os.path.exists(output_path), f"生成失敗：未找到輸出檔案 {output_path}"
size_kb = round(os.path.getsize(output_path) / 1024, 1)
print(f"[檔案大小] {size_kb} KB")
print("狀態：圖片資產生成與驗證成功！")